In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =============================================================================
# CRI-XAI PROJECT — FASE 2: PRODUZIONE SU 7.904 COMUNI
# STEP 1 — Armonizzazione regioni + Generazione popolazione sintetica
#
# APPROCCIO SCELTO: Microsimulazione (Opzione B)
# Motivazione: il dataset di training a livello individuale (df_clean) NON ha
# un identificativo di comune (solo Region/Geographic_Area), quindi non è
# possibile addestrare una regressione comune->quota_istruzione con verità nota
# a livello comunale (Opzione A). La microsimulazione invece riusa il modello
# V2 già validato (individuo -> macro-classe istruzione), applicandolo a
# individui sintetici generati coerentemente con i marginali noti del file
# comuni (fasce età×sesso, fasce di reddito con frequenza e ammontare).
#
# QUESTO STEP produce ESCLUSIVAMENTE la popolazione sintetica e i controlli di
# qualità. L'applicazione del modello V2 e l'aggregazione dei risultati per
# comune saranno lo STEP 2 (da eseguire solo dopo aver validato questo output).
# =============================================================================


# %% CELL 1 — Import e caricamento file comuni + POSAS (età singola)
# =============================================================================
# NOTA: non serve più installare hyperimpute qui — usiamo un classificatore
# standalone (RandomForest) salvato dalla Cella 17 dello script V2, non
# l'oggetto HyperImpute originale (vedi nota nella Cella 5 per il perché).
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# PLACEHOLDER: aggiorna con i path reali su Drive
PATH_COMUNI = './dataset_random_forest_completo.csv'
PATH_POSAS = './POSAS_2022_it_Comuni.csv'

df_comuni = pd.read_csv(PATH_COMUNI, sep=None, engine='python')
df_posas = pd.read_csv(PATH_POSAS, sep=';', encoding='utf-8-sig')

print(f"Comuni caricati: {len(df_comuni)}")
assert len(df_comuni) == 7904, f"Attenzione: attesi 7904 comuni, trovati {len(df_comuni)}"

# La riga Età=999 è il totale per comune (non un'età reale): la escludiamo.
# Escludiamo anche gli under-15, coerentemente col filtro "contribuenti" del
# training (nessun reddito osservabile sotto quella soglia).
df_posas_adulti = df_posas[(df_posas['Età'] != 999) & (df_posas['Età'] >= 15)].copy()

# Controllo di corrispondenza tra i codici comune dei due file — se non combaciano
# perfettamente, meglio saperlo subito che scoprirlo a metà microsimulazione.
codici_comuni = set(df_comuni['Codice.comune'].unique())
codici_posas = set(df_posas_adulti['Codice comune'].unique())
mancanti = codici_comuni - codici_posas
if mancanti:
    print(f"⚠️ {len(mancanti)} comuni presenti in df_comuni ma assenti in POSAS: {list(mancanti)[:10]}")
else:
    print(f"✅ Tutti i {len(codici_comuni)} comuni hanno una corrispondenza in POSAS (età singola).")

# Indicizziamo POSAS per comune, per accesso rapido nella Cella 4
posas_by_comune = {code: g for code, g in df_posas_adulti.groupby('Codice comune')}


# %% CELL 2 — Armonizzazione dei nomi delle regioni
# =============================================================================
# I nomi delle regioni nel file comuni NON coincidono esattamente con quelli
# usati nel training V2 (df_clean['Region']). Mappa verificata a mano:
#   - trattini mancanti ("Emilia Romagna" vs "Emilia-Romagna")
#   - Trentino-Alto Adige diviso in due "regioni" per provincia (Bolzano/Trento)
#     nel file comuni, mentre nel training è una categoria unica
#   - apostrofo tipografico diverso in "Valle d'Aosta" ('  vs  ’)
region_harmonization_map = {
    'Emilia Romagna': 'Emilia-Romagna',
    'Friuli Venezia Giulia': 'Friuli-Venezia Giulia',
    'Trentino Alto Adige(P.A.Bolzano)': 'Trentino-Alto Adige',
    'Trentino Alto Adige(P.A.Trento)': 'Trentino-Alto Adige',
    "Valle d'Aosta": 'Valle d’Aosta',  # attenzione: apostrofo tipografico a destra
}

df_comuni['Region_Harmonized'] = df_comuni['Regione'].replace(region_harmonization_map)

# Lista delle regioni valide, ESATTAMENTE come codificate nel training V2
# (stesso elenco usato in reg_map nella Cella 5 dello script V2)
valid_regions_training = [
    'Piemonte', 'Valle d’Aosta', 'Lombardia', 'Trentino-Alto Adige', 'Veneto',
    'Friuli-Venezia Giulia', 'Liguria', 'Emilia-Romagna', 'Toscana', 'Umbria',
    'Marche', 'Lazio', 'Abruzzo', 'Molise', 'Campania', 'Puglia', 'Basilicata',
    'Calabria', 'Sicilia', 'Sardegna'
]

# CONTROLLO DI INTEGRITA': se anche un solo comune non trova corrispondenza,
# ci fermiamo qui. Meglio un errore esplicito ora che una perdita di dati silenziosa.
unmatched = df_comuni[~df_comuni['Region_Harmonized'].isin(valid_regions_training)]
if len(unmatched) > 0:
    print("❌ COMUNI SENZA REGIONE ARMONIZZATA CORRETTAMENTE:")
    print(unmatched[['Comune', 'Regione', 'Region_Harmonized']].drop_duplicates())
    raise ValueError(
        f"{len(unmatched)} comuni non hanno trovato corrispondenza nella mappa regioni. "
        "Aggiorna region_harmonization_map prima di proseguire."
    )
else:
    print(f"✅ Tutti i {len(df_comuni)} comuni hanno una regione armonizzata correttamente.")

print("\nDistribuzione comuni per regione armonizzata:")
print(df_comuni['Region_Harmonized'].value_counts())


# %% CELL 3 — Preparazione dei marginali per comune (reddito) + fasce età per r_i
# =============================================================================
# L'età×sesso ora viene da POSAS (Cella 1, età singola) — non servono più le
# colonne a fasce larghe del dataset comunale per il campionamento. Le fasce
# restano utili SOLO come categoria per il calcolo di r_i (Cella 6), non più
# per generare l'età stessa.
age_bracket_edges = [15, 25, 45, 65, 150]
age_bracket_labels = ['15-24', '25-44', '45-64', '65+']

# Fasce di reddito disponibili: (colonna_frequenza, colonna_ammontare, midpoint_stimato)
# Il midpoint per l'ultima fascia aperta (>120k) è una stima prudenziale.
income_brackets = [
    ('Reddito complessivo da 0 a 10000 euro - Frequenza',
     'Reddito complessivo da 0 a 10000 euro - Ammontare in euro', 5_000),
    ('Reddito complessivo da 10000 a 15000 euro - Frequenza',
     'Reddito complessivo da 10000 a 15000 euro - Ammontare in euro', 12_500),
    ('Reddito complessivo da 15000 a 26000 euro - Frequenza',
     'Reddito complessivo da 15000 a 26000 euro - Ammontare in euro', 20_500),
    ('Reddito complessivo da 26000 a 55000 euro - Frequenza',
     'Reddito complessivo da 26000 a 55000 euro - Ammontare in euro', 40_500),
    ('Reddito complessivo da 55000 a 75000 euro - Frequenza',
     'Reddito complessivo da 55000 a 75000 euro - Ammontare in euro', 65_000),
    ('Reddito complessivo da 75000 a 120000 euro - Frequenza',
     'Reddito complessivo da 75000 a 120000 euro - Ammontare in euro', 97_500),
    ('Reddito complessivo oltre 120000 euro - Frequenza',
     'Reddito complessivo oltre 120000 euro - Ammontare in euro', 150_000),
]

# Controllo di coerenza: la somma delle frequenze per fascia di reddito dovrebbe
# essere vicina al Numero contribuenti (può non coincidere esattamente per via
# delle celle oscurate dal MEF quando i contribuenti sono < 4, come annotato nella tesi)
freq_cols = [f for f, _, _ in income_brackets]
df_comuni['_check_sum_freq'] = df_comuni[freq_cols].sum(axis=1)
scarto_medio = (df_comuni['_check_sum_freq'] - df_comuni['Numero contribuenti']).abs().mean()
print(f"\nScarto medio |somma frequenze reddito - Numero contribuenti|: {scarto_medio:.1f}")
print("(uno scarto positivo è atteso per le celle oscurate MEF sotto soglia privacy)")


# %% CELL 4 — Generazione della popolazione sintetica per comune
# =============================================================================
def genera_popolazione_sintetica(row, posas_by_comune, age_bracket_edges,
                                   age_bracket_labels, income_brackets,
                                   random_state=None):
    """
    Genera individui sintetici per UN comune, coerenti con:
      - la distribuzione età×sesso VERA e puntuale (POSAS, età singola 15-100)
      - la distribuzione di reddito (frequenza + ammontare per fascia, MEF)

    Il numero di individui sintetici generati è pari a 'Numero contribuenti'
    (i percettori di reddito), non alla popolazione totale: solo i contribuenti
    hanno un reddito osservabile nel file MEF, coerentemente con il filtro
    Total_Gross_Income > 0 applicato nel training (Cella 7 dello script V2).
    """
    rng = np.random.default_rng(random_state)
    n_contribuenti = int(row['Numero contribuenti'])
    if n_contribuenti <= 0:
        return pd.DataFrame()

    codice = row['Codice.comune']
    if codice not in posas_by_comune:
        return pd.DataFrame()  # comune senza corrispondenza POSAS: da segnalare
    posas_comune = posas_by_comune[codice]

    # --- 1. Campionamento età×sesso, dalla distribuzione VERA e puntuale POSAS ---
    # Costruiamo la lista di (età, sesso) con il relativo peso (conteggio reale)
    # NOTA: usiamo 'M'/'F' (non 'Male'/'Female') perché è così che il modello V2
    # ha effettivamente imparato a codificare il sesso — la colonna grezza
    # 'sesso' del dataset tirocinio era già testuale ('M'/'F'), quindi il
    # remapping a parole intere nello script V2 (Cella 5) non ha avuto effetto
    # e le_gender è stato fit su 'M'/'F' direttamente. Verificato con
    # le_gender.classes_ = ['F' 'M'].
    eta_vals = np.concatenate([posas_comune['Età'].values, posas_comune['Età'].values])
    sesso_vals = np.array(['M'] * len(posas_comune) + ['F'] * len(posas_comune))
    pesi = np.concatenate([posas_comune['Totale maschi'].values,
                            posas_comune['Totale femmine'].values]).astype(float)

    if pesi.sum() <= 0:
        return pd.DataFrame()  # comune senza popolazione adulta registrata: da segnalare

    probs = pesi / pesi.sum()
    sample_idx = rng.choice(len(eta_vals), size=n_contribuenti, p=probs)
    ages = eta_vals[sample_idx]
    genders = sesso_vals[sample_idx]

    # Fascia d'età derivata (serve solo per il calcolo di r_i nella Cella 6,
    # non più per il campionamento dell'età, ora puntuale)
    age_brackets_assigned = pd.cut(ages, bins=age_bracket_edges, labels=age_bracket_labels,
                                    right=False, include_lowest=True).astype(str)

    # --- 2. Campionamento reddito, proporzionale alla frequenza per fascia ---
    freqs = np.array([row[f] for f, _, _ in income_brackets], dtype=float)
    freqs = np.clip(freqs, 0, None)
    income_probs = freqs / freqs.sum() if freqs.sum() > 0 else None
    if income_probs is None:
        return pd.DataFrame()

    income_bracket_idx = rng.choice(len(income_brackets), size=n_contribuenti, p=income_probs)

    # All'interno della fascia, aggiungiamo variabilità attorno al midpoint stimato
    # (± 15%) invece di assegnare a tutti lo stesso valore, per evitare artefatti
    # nella feature Income_Age_Ratio (troppi valori identici falserebbero il modello)
    midpoints = np.array([m for _, _, m in income_brackets])[income_bracket_idx]
    incomes = midpoints * rng.uniform(0.85, 1.15, size=n_contribuenti)

    df_synth = pd.DataFrame({
        'Comune_Codice': codice,
        'Region': row['Region_Harmonized'],
        'Gender': genders,
        'Age': ages,
        'Age_Bracket': age_brackets_assigned,  # quasi-identificatore categoriale per r_i
        'Total_Gross_Income': incomes,
    })
    return df_synth


# Test su un sottoinsieme piccolo prima di lanciare su tutti i 7.904 comuni
# (generare centinaia di milioni di righe alla cieca sarebbe rischioso per la RAM)
sample_comuni = df_comuni.sample(20, random_state=RANDOM_STATE)
synth_chunks_test = []
for i, (_, row) in enumerate(sample_comuni.iterrows()):
    df_synth = genera_popolazione_sintetica(
        row, posas_by_comune, age_bracket_edges, age_bracket_labels,
        income_brackets, random_state=RANDOM_STATE + i
    )
    synth_chunks_test.append(df_synth)

df_synth_test = pd.concat(synth_chunks_test, ignore_index=True)
print(f"\n✅ Test su 20 comuni: generati {len(df_synth_test)} individui sintetici")
print(df_synth_test.head(10))
print(f"\nTotale individui sintetici stimato su tutti i 7.904 comuni: "
      f"~{int(df_comuni['Numero contribuenti'].sum()):,}")


# %% CELL 5 — Assegnazione del titolo di studio (fonte intercambiabile)
# =============================================================================
# QUESTA CELLA VA USATA COSÌ COM'È FINCHÉ FABIO NON TROVA IL DATO CENSUARIO.
# Basta cambiare EDUCATION_SOURCE quando arriva il file nuovo — nessun'altra
# riga dello script cambia.
#
# Due modalità:
#   "hyperimpute"  -> applica il modello V2 già validato (age/gender/region/income
#                     di ogni individuo sintetico) per PREDIRE l'istruzione.
#                     Più ricco (usa tutte le covariate), ma è una STIMA con
#                     margine d'errore (~57% di accuratezza validata).
#   "censimento"   -> legge le proporzioni VERE per comune da un file esterno
#                     (quello che Fabio sta cercando) e campiona secondo quella
#                     quota. Meno ricco (solo marginale comunale), ma osservato,
#                     non stimato.
EDUCATION_SOURCE = "hyperimpute"  # cambia in "censimento" quando arriva il file

# --- Path del file censuario, da aggiornare quando Fabio lo trova ---
PATH_CENSIMENTO_ISTRUZIONE = '/content/drive/MyDrive/Paper ASMBI/Dataset/dataset_titolo_studio.xlsx'

# Atteso: colonne Codice.comune, quota_istruzione_0_Base, quota_istruzione_1_Medio,
# quota_istruzione_2_Alto (le tre proporzioni sommano a 1 per ogni comune)

# --- Path del pacchetto modello+encoder già addestrato e "fissato" ---
PATH_MODELLO_V2 = '/content/drive/MyDrive/Paper ASMBI/Modelli/hyperimpute_v2_standalone_classifier.joblib'
# NOTA: usiamo il classificatore standalone (Cella 17 dello script V2), non
# l'oggetto HyperImpute originale — quest'ultimo non espone un metodo .predict()
# utilizzabile su record nuovi, ed è comunque troppo lento per 41 milioni di
# individui sintetici (rilancerebbe l'intera ricerca AutoML ad ogni chiamata).
# Essendo un RandomForestClassifier normale, si carica con joblib — niente più
# bisogno di cloudpickle né di installare hyperimpute in questo notebook.
import joblib
pacchetto_v2 = joblib.load(PATH_MODELLO_V2)
modello_v2 = pacchetto_v2['model']
le_gender = pacchetto_v2['le_gender']
le_region = pacchetto_v2['le_region']
print("✅ Modello V2 (standalone) ed encoder caricati correttamente.")
print(f"   Categorie Gender conosciute dal modello: {list(le_gender.classes_)}")
print(f"   Categorie Region conosciute dal modello: {list(le_region.classes_)}")

macro_names = ['0_Base', '1_Medio', '2_Alto']


def assegna_istruzione_hyperimpute(df_synth, modello_v2, le_gender, le_region):
    """
    Applica il modello V2 (già validato, ~57% accuratezza) a ciascun individuo
    sintetico per predire la macro-classe di istruzione, usando le stesse
    feature del training: Gender, Age, Region, Total_Gross_Income, Income_Age_Ratio.
    """
    df_synth = df_synth.copy()

    # Controllo preventivo: se generiamo categorie che il modello non conosce,
    # meglio un errore chiaro qui che il messaggio criptico di sklearn più sotto.
    gender_ignoti = set(df_synth['Gender'].unique()) - set(le_gender.classes_)
    region_ignote = set(df_synth['Region'].unique()) - set(le_region.classes_)
    if gender_ignoti:
        raise ValueError(f"Gender non riconosciuti dal modello: {gender_ignoti}. "
                          f"Il modello conosce solo: {list(le_gender.classes_)}")
    if region_ignote:
        raise ValueError(f"Region non riconosciute dal modello: {region_ignote}. "
                          f"Il modello conosce solo: {list(le_region.classes_)}")

    df_synth['Income_Age_Ratio'] = df_synth['Total_Gross_Income'] / (df_synth['Age'] + 1)
    X = pd.DataFrame({
        'Gender': le_gender.transform(df_synth['Gender'].astype(str)),
        'Age': df_synth['Age'],
        'Region': le_region.transform(df_synth['Region'].astype(str)),
        'Total_Gross_Income': df_synth['Total_Gross_Income'],
        'Income_Age_Ratio': df_synth['Income_Age_Ratio'],
    })
    pred_idx = modello_v2.predict(X)
    df_synth['Education_Macro'] = [macro_names[i] for i in pred_idx]
    return df_synth


def assegna_istruzione_censimento(df_synth, comune_row):
    """
    Campiona l'istruzione per gli individui sintetici di UN comune secondo le
    proporzioni VERE lette dal file censuario — indipendente dalle altre
    covariate individuali, perché il censimento dà solo la quota aggregata
    per comune, non l'incrocio con età/sesso/reddito.
    """
    rng = np.random.default_rng()
    quote = np.array([
        comune_row['quota_istruzione_0_Base'],
        comune_row['quota_istruzione_1_Medio'],
        comune_row['quota_istruzione_2_Alto'],
    ], dtype=float)
    quote = quote / quote.sum()  # sicurezza: rinormalizza se non sommano esattamente a 1
    idx = rng.choice(3, size=len(df_synth), p=quote)
    df_synth = df_synth.copy()
    df_synth['Education_Macro'] = [macro_names[i] for i in idx]
    return df_synth


print(f"Fonte istruzione attiva: {EDUCATION_SOURCE}")
print("Ricorda di cambiare EDUCATION_SOURCE in 'censimento' non appena Fabio "
      "trova il file — nessun'altra riga di questo script va toccata.")

# --- Applicazione al test dei 20 comuni, per vedere subito un risultato concreto ---
if EDUCATION_SOURCE == "hyperimpute":
    df_synth_test = assegna_istruzione_hyperimpute(df_synth_test, modello_v2, le_gender, le_region)
else:
    # ramo censimento: da applicare comune per comune, non su tutto il blocco insieme
    raise NotImplementedError("Applica assegna_istruzione_censimento per singolo comune quando arriva il file")

print("\n✅ Istruzione assegnata al test dei 20 comuni:")
print(df_synth_test['Education_Macro'].value_counts())
print(df_synth_test.head(10))


# %% CELL 6 — Calcolo di r_i (Młodak) sulla popolazione sintetica
# =============================================================================
def calcola_r_i(df_synth, p_income=0.10, k_mef=4, lam=0.5):
    """
    Calcola il rischio individuale di Młodak per ogni individuo sintetico,
    DENTRO IL SUO COMUNE (i "vicini" si contano solo tra individui dello
    stesso comune, coerente con l'aggregazione territoriale del paper).

    Quasi-identificatori categoriali (h=3): Gender, Age_Bracket, Education_Macro
    Feature continua fiscale (m-h=1): Total_Gross_Income, soglia k=4 (MEF)

    p_income: precisione dell'intervallo di vicinanza sul reddito (paper: p_j)
    k_mef: soglia di sicurezza MEF (paper: k=4 per dati fiscali)
    lam: peso di bilanciamento identity/attribute disclosure (paper: lambda=0.5)
    """
    df = df_synth.copy()

    # --- 1. f_i: frequenza della combinazione categoriale completa, PER COMUNE ---
    combo_cols = ['Comune_Codice', 'Gender', 'Age_Bracket', 'Education_Macro']
    df['f_i'] = df.groupby(combo_cols)['Comune_Codice'].transform('count')

    # --- 2. w_ij: vicinanza sul reddito, PER COMUNE (ordinamento + ricerca binaria,
    #      per evitare un confronto O(n^2) su milioni di record) ---
    w_list = np.empty(len(df), dtype=int)
    for comune_code, idx in df.groupby('Comune_Codice').groups.items():
        idx = np.array(idx)
        incomes = df.loc[idx, 'Total_Gross_Income'].values
        order = np.argsort(incomes)
        sorted_incomes = incomes[order]
        n = len(sorted_incomes)

        lower = sorted_incomes / (1 + p_income)
        upper = sorted_incomes * (1 + p_income)
        # numero di osservazioni (incluso se stesso) nell'intervallo, poi -1 per escludersi
        left = np.searchsorted(sorted_incomes, lower, side='left')
        right = np.searchsorted(sorted_incomes, upper, side='right')
        eta = (right - left) - 1  # esclude l'osservazione stessa, come da formula

        w_sorted = (eta < k_mef).astype(int)  # w_ij = 1 se isolato (rischio), 0 se protetto
        w_orig_order = np.empty(n, dtype=int)
        w_orig_order[order] = w_sorted
        w_list[idx] = w_orig_order

    df['w_income'] = w_list

    # --- 3. r_i = lambda*(1/f_i) + (1-lambda)*(media dei w_ij sulle feature continue) ---
    # Con una sola feature continua (Total_Gross_Income), la media è w_income stesso.
    # Se in futuro si aggiungono altre variabili fiscali continue, estendere qui.
    df['r_i'] = lam * (1 / df['f_i']) + (1 - lam) * df['w_income']

    return df


print("Funzione calcola_r_i pronta. Verrà applicata dopo aver generato e assegnato")
print("l'istruzione all'intera popolazione sintetica (Celle 4 + 5), non solo al test da 20 comuni.")

# --- Applicazione al test dei 20 comuni ---
df_synth_test = calcola_r_i(df_synth_test)
print("\n✅ r_i calcolato sul test dei 20 comuni:")
print(df_synth_test[['Comune_Codice', 'Gender', 'Age', 'Education_Macro',
                      'Total_Gross_Income', 'f_i', 'w_income', 'r_i']].head(10))
print("\nDistribuzione di r_i (dovrebbe stare tra 0 e 1):")
print(df_synth_test['r_i'].describe())

Comuni caricati: 7904
✅ Tutti i 7904 comuni hanno una corrispondenza in POSAS (età singola).
✅ Tutti i 7904 comuni hanno una regione armonizzata correttamente.

Distribuzione comuni per regione armonizzata:
Region_Harmonized
Lombardia                1506
Piemonte                 1181
Veneto                    563
Campania                  550
Calabria                  404
Sicilia                   391
Lazio                     378
Sardegna                  377
Emilia-Romagna            330
Abruzzo                   305
Trentino-Alto Adige       282
Toscana                   273
Puglia                    257
Liguria                   234
Marche                    225
Friuli-Venezia Giulia     215
Molise                    136
Basilicata                131
Umbria                     92
Valle d’Aosta              74
Name: count, dtype: int64

Scarto medio |somma frequenze reddito - Numero contribuenti|: 130.3
(uno scarto positivo è atteso per le celle oscurate MEF sotto soglia privacy)

✅

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Paper ASMBI/Modelli/hyperimpute_v2_standalone_classifier.joblib'

fino a qui era una prova per vedere come si comportava il modello ora estendiamo il alvoro a tutti i comuni

In [ ]:
# =============================================================================
# CRI-XAI PROJECT — FASE 2: PRODUZIONE SU 7.904 COMUNI
# STEP 1 — Armonizzazione regioni + Generazione popolazione sintetica
#
# APPROCCIO SCELTO: Microsimulazione (Opzione B)
# Motivazione: il dataset di training a livello individuale (df_clean) NON ha
# un identificativo di comune (solo Region/Geographic_Area), quindi non è
# possibile addestrare una regressione comune->quota_istruzione con verità nota
# a livello comunale (Opzione A). La microsimulazione invece riusa il modello
# V2 già validato (individuo -> macro-classe istruzione), applicandolo a
# individui sintetici generati coerentemente con i marginali noti del file
# comuni (fasce età×sesso, fasce di reddito con frequenza e ammontare).
#
# QUESTO STEP produce ESCLUSIVAMENTE la popolazione sintetica e i controlli di
# qualità. L'applicazione del modello V2 e l'aggregazione dei risultati per
# comune saranno lo STEP 2 (da eseguire solo dopo aver validato questo output).
# =============================================================================


# %% CELL 1 — Import e caricamento file comuni + POSAS (età singola)
# =============================================================================
# NOTA SUGLI ANNI DI RIFERIMENTO (verificato su fonte ISTAT):
# POSAS "2022" NON è un dato del 2022 — ISTAT calcola la popolazione residente
# al 31 dicembre di ogni anno e la diffonde il 1° gennaio dell'anno successivo.
# Quindi POSAS_2022 fotografa la popolazione al 31/12/2021, esattamente lo
# stesso periodo di riferimento dei redditi MEF 2021 e del dataset tirocinio
# "completo 2021". Nessun disallineamento tra le fonti: è solo una convenzione
# di etichettatura (anno di pubblicazione, non anno di riferimento).
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# PLACEHOLDER: aggiorna con i path reali su Drive
PATH_COMUNI = '/content/drive/MyDrive/Paper ASMBI/Dataset/dataset_random_forest_completo.csv'
PATH_POSAS = '/content/drive/MyDrive/Paper ASMBI/Dataset/POSAS_2022_it_Comuni.csv'

df_comuni = pd.read_csv(PATH_COMUNI, sep=None, engine='python')
df_posas = pd.read_csv(PATH_POSAS, sep=';', encoding='utf-8-sig')

print(f"Comuni caricati: {len(df_comuni)}")
assert len(df_comuni) == 7904, f"Attenzione: attesi 7904 comuni, trovati {len(df_comuni)}"

# La riga Età=999 è il totale per comune (non un'età reale): la escludiamo.
# Escludiamo anche gli under-15, coerentemente col filtro "contribuenti" del
# training (nessun reddito osservabile sotto quella soglia).
df_posas_adulti = df_posas[(df_posas['Età'] != 999) & (df_posas['Età'] >= 15)].copy()

# Controllo di corrispondenza tra i codici comune dei due file — se non combaciano
# perfettamente, meglio saperlo subito che scoprirlo a metà microsimulazione.
codici_comuni = set(df_comuni['Codice.comune'].unique())
codici_posas = set(df_posas_adulti['Codice comune'].unique())
mancanti = codici_comuni - codici_posas
if mancanti:
    print(f"⚠️ {len(mancanti)} comuni presenti in df_comuni ma assenti in POSAS: {list(mancanti)[:10]}")
else:
    print(f"✅ Tutti i {len(codici_comuni)} comuni hanno una corrispondenza in POSAS (età singola).")

# Indicizziamo POSAS per comune, per accesso rapido nella Cella 4
posas_by_comune = {code: g for code, g in df_posas_adulti.groupby('Codice comune')}


# %% CELL 2 — Armonizzazione dei nomi delle regioni
# =============================================================================
# I nomi delle regioni nel file comuni NON coincidono esattamente con quelli
# usati nel training V2 (df_clean['Region']). Mappa verificata a mano:
#   - trattini mancanti ("Emilia Romagna" vs "Emilia-Romagna")
#   - Trentino-Alto Adige diviso in due "regioni" per provincia (Bolzano/Trento)
#     nel file comuni, mentre nel training è una categoria unica
#   - apostrofo tipografico diverso in "Valle d'Aosta" ('  vs  ’)
region_harmonization_map = {
    'Emilia Romagna': 'Emilia-Romagna',
    'Friuli Venezia Giulia': 'Friuli-Venezia Giulia',
    'Trentino Alto Adige(P.A.Bolzano)': 'Trentino-Alto Adige',
    'Trentino Alto Adige(P.A.Trento)': 'Trentino-Alto Adige',
    "Valle d'Aosta": 'Valle d’Aosta',  # attenzione: apostrofo tipografico a destra
}

df_comuni['Region_Harmonized'] = df_comuni['Regione'].replace(region_harmonization_map)

# Lista delle regioni valide, ESATTAMENTE come codificate nel training V2
# (stesso elenco usato in reg_map nella Cella 5 dello script V2)
valid_regions_training = [
    'Piemonte', 'Valle d’Aosta', 'Lombardia', 'Trentino-Alto Adige', 'Veneto',
    'Friuli-Venezia Giulia', 'Liguria', 'Emilia-Romagna', 'Toscana', 'Umbria',
    'Marche', 'Lazio', 'Abruzzo', 'Molise', 'Campania', 'Puglia', 'Basilicata',
    'Calabria', 'Sicilia', 'Sardegna'
]

# CONTROLLO DI INTEGRITA': se anche un solo comune non trova corrispondenza,
# ci fermiamo qui. Meglio un errore esplicito ora che una perdita di dati silenziosa.
unmatched = df_comuni[~df_comuni['Region_Harmonized'].isin(valid_regions_training)]
if len(unmatched) > 0:
    print("❌ COMUNI SENZA REGIONE ARMONIZZATA CORRETTAMENTE:")
    print(unmatched[['Comune', 'Regione', 'Region_Harmonized']].drop_duplicates())
    raise ValueError(
        f"{len(unmatched)} comuni non hanno trovato corrispondenza nella mappa regioni. "
        "Aggiorna region_harmonization_map prima di proseguire."
    )
else:
    print(f"✅ Tutti i {len(df_comuni)} comuni hanno una regione armonizzata correttamente.")

print("\nDistribuzione comuni per regione armonizzata:")
print(df_comuni['Region_Harmonized'].value_counts())


# %% CELL 3 — Preparazione dei marginali per comune (reddito) + fasce età per r_i
# =============================================================================
# L'età×sesso ora viene da POSAS (Cella 1, età singola) — non servono più le
# colonne a fasce larghe del dataset comunale per il campionamento. Le fasce
# restano utili SOLO come categoria per il calcolo di r_i (Cella 6), non più
# per generare l'età stessa.
age_bracket_edges = [15, 25, 45, 65, 150]
age_bracket_labels = ['15-24', '25-44', '45-64', '65+']

# Fasce di reddito disponibili: (colonna_frequenza, colonna_ammontare, midpoint_stimato)
# Il midpoint per l'ultima fascia aperta (>120k) è una stima prudenziale.
income_brackets = [
    ('Reddito complessivo da 0 a 10000 euro - Frequenza',
     'Reddito complessivo da 0 a 10000 euro - Ammontare in euro', 5_000),
    ('Reddito complessivo da 10000 a 15000 euro - Frequenza',
     'Reddito complessivo da 10000 a 15000 euro - Ammontare in euro', 12_500),
    ('Reddito complessivo da 15000 a 26000 euro - Frequenza',
     'Reddito complessivo da 15000 a 26000 euro - Ammontare in euro', 20_500),
    ('Reddito complessivo da 26000 a 55000 euro - Frequenza',
     'Reddito complessivo da 26000 a 55000 euro - Ammontare in euro', 40_500),
    ('Reddito complessivo da 55000 a 75000 euro - Frequenza',
     'Reddito complessivo da 55000 a 75000 euro - Ammontare in euro', 65_000),
    ('Reddito complessivo da 75000 a 120000 euro - Frequenza',
     'Reddito complessivo da 75000 a 120000 euro - Ammontare in euro', 97_500),
    ('Reddito complessivo oltre 120000 euro - Frequenza',
     'Reddito complessivo oltre 120000 euro - Ammontare in euro', 150_000),
]

# Controllo di coerenza: la somma delle frequenze per fascia di reddito dovrebbe
# essere vicina al Numero contribuenti (può non coincidere esattamente per via
# delle celle oscurate dal MEF quando i contribuenti sono < 4, come annotato nella tesi)
freq_cols = [f for f, _, _ in income_brackets]
df_comuni['_check_sum_freq'] = df_comuni[freq_cols].sum(axis=1)
scarto_medio = (df_comuni['_check_sum_freq'] - df_comuni['Numero contribuenti']).abs().mean()
print(f"\nScarto medio |somma frequenze reddito - Numero contribuenti|: {scarto_medio:.1f}")
print("(uno scarto positivo è atteso per le celle oscurate MEF sotto soglia privacy)")


# %% CELL 4 — Generazione della popolazione sintetica per comune
# =============================================================================
def genera_popolazione_sintetica(row, posas_by_comune, age_bracket_edges,
                                   age_bracket_labels, income_brackets,
                                   random_state=None):
    """
    Genera individui sintetici per UN comune, coerenti con:
      - la distribuzione età×sesso VERA e puntuale (POSAS, età singola 15-100)
      - la distribuzione di reddito (frequenza + ammontare per fascia, MEF)

    Il numero di individui sintetici generati è pari a 'Numero contribuenti'
    (i percettori di reddito), non alla popolazione totale: solo i contribuenti
    hanno un reddito osservabile nel file MEF, coerentemente con il filtro
    Total_Gross_Income > 0 applicato nel training (Cella 7 dello script V2).
    """
    rng = np.random.default_rng(random_state)
    n_contribuenti = int(row['Numero contribuenti'])
    if n_contribuenti <= 0:
        return pd.DataFrame()

    codice = row['Codice.comune']
    if codice not in posas_by_comune:
        return pd.DataFrame()  # comune senza corrispondenza POSAS: da segnalare
    posas_comune = posas_by_comune[codice]

    # --- 1. Campionamento età×sesso, dalla distribuzione VERA e puntuale POSAS ---
    # Costruiamo la lista di (età, sesso) con il relativo peso (conteggio reale)
    # NOTA: usiamo 'M'/'F' (non 'Male'/'Female') perché è così che il modello V2
    # ha effettivamente imparato a codificare il sesso — la colonna grezza
    # 'sesso' del dataset tirocinio era già testuale ('M'/'F'), quindi il
    # remapping a parole intere nello script V2 (Cella 5) non ha avuto effetto
    # e le_gender è stato fit su 'M'/'F' direttamente. Verificato con
    # le_gender.classes_ = ['F' 'M'].
    eta_vals = np.concatenate([posas_comune['Età'].values, posas_comune['Età'].values])
    sesso_vals = np.array(['M'] * len(posas_comune) + ['F'] * len(posas_comune))
    pesi = np.concatenate([posas_comune['Totale maschi'].values,
                            posas_comune['Totale femmine'].values]).astype(float)

    if pesi.sum() <= 0:
        return pd.DataFrame()  # comune senza popolazione adulta registrata: da segnalare

    probs = pesi / pesi.sum()
    sample_idx = rng.choice(len(eta_vals), size=n_contribuenti, p=probs)
    ages = eta_vals[sample_idx]
    genders = sesso_vals[sample_idx]

    # Fascia d'età derivata (serve solo per il calcolo di r_i nella Cella 6,
    # non più per il campionamento dell'età, ora puntuale)
    age_brackets_assigned = pd.cut(ages, bins=age_bracket_edges, labels=age_bracket_labels,
                                    right=False, include_lowest=True).astype(str)

    # --- 2. Campionamento reddito, proporzionale alla frequenza per fascia ---
    freqs = np.array([row[f] for f, _, _ in income_brackets], dtype=float)
    # CORREZIONE: le fasce di reddito alto sono spesso NaN per i comuni piccoli
    # (oscurate da MEF sotto soglia di privacy) — np.clip() NON converte NaN in 0,
    # quindi senza questa riga freqs.sum() diventa NaN e l'INTERO comune veniva
    # scartato (bug trovato dopo la prima run: 2.684 comuni persi su 7.904,
    # sistematicamente i più piccoli — proprio quelli centrali per il paper)
    freqs = np.nan_to_num(freqs, nan=0.0)
    freqs = np.clip(freqs, 0, None)
    income_probs = freqs / freqs.sum() if freqs.sum() > 0 else None
    if income_probs is None:
        return pd.DataFrame()

    income_bracket_idx = rng.choice(len(income_brackets), size=n_contribuenti, p=income_probs)

    # All'interno della fascia, aggiungiamo variabilità attorno al midpoint stimato
    # (± 15%) invece di assegnare a tutti lo stesso valore, per evitare artefatti
    # nella feature Income_Age_Ratio (troppi valori identici falserebbero il modello)
    midpoints = np.array([m for _, _, m in income_brackets])[income_bracket_idx]
    incomes = midpoints * rng.uniform(0.85, 1.15, size=n_contribuenti)

    df_synth = pd.DataFrame({
        'Comune_Codice': codice,
        'Region': row['Region_Harmonized'],
        'Gender': genders,
        'Age': ages,
        'Age_Bracket': age_brackets_assigned,  # quasi-identificatore categoriale per r_i
        'Total_Gross_Income': incomes,
    })
    return df_synth


# Test su un sottoinsieme piccolo prima di lanciare su tutti i 7.904 comuni
# (generare centinaia di milioni di righe alla cieca sarebbe rischioso per la RAM)
sample_comuni = df_comuni.sample(20, random_state=RANDOM_STATE)
synth_chunks_test = []
for i, (_, row) in enumerate(sample_comuni.iterrows()):
    df_synth = genera_popolazione_sintetica(
        row, posas_by_comune, age_bracket_edges, age_bracket_labels,
        income_brackets, random_state=RANDOM_STATE + i
    )
    synth_chunks_test.append(df_synth)

df_synth_test = pd.concat(synth_chunks_test, ignore_index=True)
print(f"\n✅ Test su 20 comuni: generati {len(df_synth_test)} individui sintetici")
print(df_synth_test.head(10))
print(f"\nTotale individui sintetici stimato su tutti i 7.904 comuni: "
      f"~{int(df_comuni['Numero contribuenti'].sum()):,}")


# %% CELL 5 — Assegnazione del titolo di studio (fonte intercambiabile)
# =============================================================================
# QUESTA CELLA VA USATA COSÌ COM'È FINCHÉ FABIO NON TROVA IL DATO CENSUARIO.
# Basta cambiare EDUCATION_SOURCE quando arriva il file nuovo — nessun'altra
# riga dello script cambia.
#
# Due modalità:
#   "hyperimpute"  -> applica il modello V2 già validato (age/gender/region/income
#                     di ogni individuo sintetico) per PREDIRE l'istruzione.
#                     Più ricco (usa tutte le covariate), ma è una STIMA con
#                     margine d'errore (~57% di accuratezza validata).
#   "censimento"   -> legge le proporzioni VERE per comune da un file esterno
#                     (quello che Fabio sta cercando) e campiona secondo quella
#                     quota. Meno ricco (solo marginale comunale), ma osservato,
#                     non stimato.
EDUCATION_SOURCE = "hyperimpute"  # cambia in "censimento" quando arriva il file

# --- Path del file censuario, da aggiornare quando Fabio lo trova ---
PATH_CENSIMENTO_ISTRUZIONE = '/content/drive/MyDrive/Paper ASMBI/Dataset/censimento_istruzione_comuni.csv'
# Atteso: colonne Codice.comune, quota_istruzione_0_Base, quota_istruzione_1_Medio,
# quota_istruzione_2_Alto (le tre proporzioni sommano a 1 per ogni comune)

# --- Path del pacchetto modello+encoder già addestrato e "fissato" ---
PATH_MODELLO_V2 = '/content/drive/MyDrive/Paper ASMBI/Modelli/hyperimpute_v2_standalone_classifier.joblib'
# NOTA: usiamo il classificatore standalone (Cella 17 dello script V2), non
# l'oggetto HyperImpute originale — quest'ultimo non espone un metodo .predict()
# utilizzabile su record nuovi, ed è comunque troppo lento per 41 milioni di
# individui sintetici (rilancerebbe l'intera ricerca AutoML ad ogni chiamata).
# Essendo un RandomForestClassifier normale, si carica con joblib — niente più
# bisogno di cloudpickle né di installare hyperimpute in questo notebook.
import joblib
pacchetto_v2 = joblib.load(PATH_MODELLO_V2)
modello_v2 = pacchetto_v2['model']
le_gender = pacchetto_v2['le_gender']
le_region = pacchetto_v2['le_region']
print("✅ Modello V2 (standalone) ed encoder caricati correttamente.")
print(f"   Categorie Gender conosciute dal modello: {list(le_gender.classes_)}")
print(f"   Categorie Region conosciute dal modello: {list(le_region.classes_)}")

macro_names = ['0_Base', '1_Medio', '2_Alto']


def assegna_istruzione_hyperimpute(df_synth, modello_v2, le_gender, le_region):
    """
    Applica il modello V2 (già validato, ~57% accuratezza) a ciascun individuo
    sintetico per predire la macro-classe di istruzione, usando le stesse
    feature del training: Gender, Age, Region, Total_Gross_Income, Income_Age_Ratio.
    """
    df_synth = df_synth.copy()

    # Controllo preventivo: se generiamo categorie che il modello non conosce,
    # meglio un errore chiaro qui che il messaggio criptico di sklearn più sotto.
    gender_ignoti = set(df_synth['Gender'].unique()) - set(le_gender.classes_)
    region_ignote = set(df_synth['Region'].unique()) - set(le_region.classes_)
    if gender_ignoti:
        raise ValueError(f"Gender non riconosciuti dal modello: {gender_ignoti}. "
                          f"Il modello conosce solo: {list(le_gender.classes_)}")
    if region_ignote:
        raise ValueError(f"Region non riconosciute dal modello: {region_ignote}. "
                          f"Il modello conosce solo: {list(le_region.classes_)}")

    df_synth['Income_Age_Ratio'] = df_synth['Total_Gross_Income'] / (df_synth['Age'] + 1)
    X = pd.DataFrame({
        'Gender': le_gender.transform(df_synth['Gender'].astype(str)),
        'Age': df_synth['Age'],
        'Region': le_region.transform(df_synth['Region'].astype(str)),
        'Total_Gross_Income': df_synth['Total_Gross_Income'],
        'Income_Age_Ratio': df_synth['Income_Age_Ratio'],
    })
    pred_idx = modello_v2.predict(X)
    df_synth['Education_Macro'] = [macro_names[i] for i in pred_idx]
    return df_synth


def assegna_istruzione_censimento(df_synth, comune_row):
    """
    Campiona l'istruzione per gli individui sintetici di UN comune secondo le
    proporzioni VERE lette dal file censuario — indipendente dalle altre
    covariate individuali, perché il censimento dà solo la quota aggregata
    per comune, non l'incrocio con età/sesso/reddito.
    """
    rng = np.random.default_rng()
    quote = np.array([
        comune_row['quota_istruzione_0_Base'],
        comune_row['quota_istruzione_1_Medio'],
        comune_row['quota_istruzione_2_Alto'],
    ], dtype=float)
    quote = quote / quote.sum()  # sicurezza: rinormalizza se non sommano esattamente a 1
    idx = rng.choice(3, size=len(df_synth), p=quote)
    df_synth = df_synth.copy()
    df_synth['Education_Macro'] = [macro_names[i] for i in idx]
    return df_synth


print(f"Fonte istruzione attiva: {EDUCATION_SOURCE}")
print("Ricorda di cambiare EDUCATION_SOURCE in 'censimento' non appena Fabio "
      "trova il file — nessun'altra riga di questo script va toccata.")

# --- Applicazione al test dei 20 comuni, per vedere subito un risultato concreto ---
if EDUCATION_SOURCE == "hyperimpute":
    df_synth_test = assegna_istruzione_hyperimpute(df_synth_test, modello_v2, le_gender, le_region)
else:
    # ramo censimento: da applicare comune per comune, non su tutto il blocco insieme
    raise NotImplementedError("Applica assegna_istruzione_censimento per singolo comune quando arriva il file")

print("\n✅ Istruzione assegnata al test dei 20 comuni:")
print(df_synth_test['Education_Macro'].value_counts())
print(df_synth_test.head(10))


# %% CELL 6 — Calcolo di r_i (Młodak) sulla popolazione sintetica
# =============================================================================
def calcola_r_i(df_synth, p_income=0.10, k_mef=4, lam=0.5):
    """
    Calcola il rischio individuale di Młodak per ogni individuo sintetico,
    DENTRO IL SUO COMUNE (i "vicini" si contano solo tra individui dello
    stesso comune, coerente con l'aggregazione territoriale del paper).

    Quasi-identificatori categoriali (h=3): Gender, Age_Bracket, Education_Macro
    Feature continua fiscale (m-h=1): Total_Gross_Income, soglia k=4 (MEF)

    p_income: precisione dell'intervallo di vicinanza sul reddito (paper: p_j)
    k_mef: soglia di sicurezza MEF (paper: k=4 per dati fiscali)
    lam: peso di bilanciamento identity/attribute disclosure (paper: lambda=0.5)
    """
    df = df_synth.copy()

    # --- 1. f_i: frequenza della combinazione categoriale completa, PER COMUNE ---
    combo_cols = ['Comune_Codice', 'Gender', 'Age_Bracket', 'Education_Macro']
    df['f_i'] = df.groupby(combo_cols)['Comune_Codice'].transform('count')

    # --- 2. w_ij: vicinanza sul reddito, PER COMUNE (ordinamento + ricerca binaria,
    #      per evitare un confronto O(n^2) su milioni di record) ---
    w_list = np.empty(len(df), dtype=int)
    for comune_code, idx in df.groupby('Comune_Codice').groups.items():
        idx = np.array(idx)
        incomes = df.loc[idx, 'Total_Gross_Income'].values
        order = np.argsort(incomes)
        sorted_incomes = incomes[order]
        n = len(sorted_incomes)

        lower = sorted_incomes / (1 + p_income)
        upper = sorted_incomes * (1 + p_income)
        # numero di osservazioni (incluso se stesso) nell'intervallo, poi -1 per escludersi
        left = np.searchsorted(sorted_incomes, lower, side='left')
        right = np.searchsorted(sorted_incomes, upper, side='right')
        eta = (right - left) - 1  # esclude l'osservazione stessa, come da formula

        w_sorted = (eta < k_mef).astype(int)  # w_ij = 1 se isolato (rischio), 0 se protetto
        w_orig_order = np.empty(n, dtype=int)
        w_orig_order[order] = w_sorted
        w_list[idx] = w_orig_order

    df['w_income'] = w_list

    # --- 3. r_i = lambda*(1/f_i) + (1-lambda)*(media dei w_ij sulle feature continue) ---
    # Con una sola feature continua (Total_Gross_Income), la media è w_income stesso.
    # Se in futuro si aggiungono altre variabili fiscali continue, estendere qui.
    df['r_i'] = lam * (1 / df['f_i']) + (1 - lam) * df['w_income']

    return df


print("Funzione calcola_r_i pronta. Verrà applicata dopo aver generato e assegnato")
print("l'istruzione all'intera popolazione sintetica (Celle 4 + 5), non solo al test da 20 comuni.")

# --- Applicazione al test dei 20 comuni ---
df_synth_test = calcola_r_i(df_synth_test)
print("\n✅ r_i calcolato sul test dei 20 comuni:")
print(df_synth_test[['Comune_Codice', 'Gender', 'Age', 'Education_Macro',
                      'Total_Gross_Income', 'f_i', 'w_income', 'r_i']].head(10))
print("\nDistribuzione di r_i (dovrebbe stare tra 0 e 1):")
print(df_synth_test['r_i'].describe())


# %% CELL 7 — GENERAZIONE COMPLETA SU TUTTI I 7.904 COMUNI
# =============================================================================
# Con ~41 milioni di individui sintetici attesi, NON teniamo tutto in memoria:
#   1. Elaboriamo un blocco di comuni alla volta (BATCH_SIZE)
#   2. Per ogni comune salviamo SOLO l'aggregato (proporzioni di istruzione),
#      non i singoli individui — evita di accumulare 41M righe in RAM
#   3. Teniamo un CAMPIONE individuale limitato (non tutti i 41M) per il futuro
#      addestramento del nuovo R_i — una dimensione del genere è già più che
#      sufficiente per un Random Forest, non serve la popolazione intera
#   4. Salviamo su Drive OGNI BATCH, non solo alla fine — se Colab si disconnette
#      (è già successo diverse volte in questo progetto), riprendi dal
#      checkpoint invece di perdere tutto il lavoro

import os

BATCH_SIZE = 200          # comuni processati per blocco prima di salvare
CAMPIONE_PER_COMUNE = 150  # individui sintetici conservati per comune (per R_i)

PATH_QUOTE_OUTPUT = '/content/drive/MyDrive/Paper ASMBI/Dataset/quota_istruzione_comuni.csv'
PATH_CAMPIONE_OUTPUT = '/content/drive/MyDrive/Paper ASMBI/Dataset/campione_individuale_r_i.csv'
PATH_CHECKPOINT = '/content/drive/MyDrive/Paper ASMBI/Dataset/_checkpoint_microsimulazione.txt'

# --- Ripresa da checkpoint, se esiste ---
comuni_gia_fatti = 0
if os.path.exists(PATH_CHECKPOINT):
    with open(PATH_CHECKPOINT, 'r') as f:
        comuni_gia_fatti = int(f.read().strip())
    print(f"⏯️ Ripresa da checkpoint: {comuni_gia_fatti} comuni già processati, continuo da lì.")
else:
    print("▶️ Nessun checkpoint trovato, parto da zero.")
    # Se ripartiamo da zero, puliamo eventuali file parziali di run precedenti falliti
    for p in [PATH_QUOTE_OUTPUT, PATH_CAMPIONE_OUTPUT]:
        if os.path.exists(p):
            os.remove(p)

df_comuni_da_fare = df_comuni.iloc[comuni_gia_fatti:].reset_index(drop=True)
n_batch_totali = (len(df_comuni_da_fare) // BATCH_SIZE) + 1

for batch_idx in range(n_batch_totali):
    batch = df_comuni_da_fare.iloc[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]
    if len(batch) == 0:
        break

    quote_batch = []
    campioni_batch = []

    for i, (_, row) in enumerate(batch.iterrows()):
        seed_comune = RANDOM_STATE + comuni_gia_fatti + batch_idx * BATCH_SIZE + i
        df_synth = genera_popolazione_sintetica(
            row, posas_by_comune, age_bracket_edges, age_bracket_labels,
            income_brackets, random_state=seed_comune
        )
        if len(df_synth) == 0:
            continue  # comune senza dati sufficienti, già segnalato dalla funzione

        df_synth = assegna_istruzione_hyperimpute(df_synth, modello_v2, le_gender, le_region)
        df_synth = calcola_r_i(df_synth)

        # Aggregato per M_i / P_i: proporzioni di istruzione per questo comune
        quote = df_synth['Education_Macro'].value_counts(normalize=True)
        quote_batch.append({
            'Codice.comune': row['Codice.comune'],
            'quota_istruzione_0_Base': quote.get('0_Base', 0.0),
            'quota_istruzione_1_Medio': quote.get('1_Medio', 0.0),
            'quota_istruzione_2_Alto': quote.get('2_Alto', 0.0),
            'n_individui_sintetici': len(df_synth),
        })

        # Campione individuale limitato per il futuro training di R_i
        n_campione = min(CAMPIONE_PER_COMUNE, len(df_synth))
        campioni_batch.append(df_synth.sample(n_campione, random_state=seed_comune))

    # --- Salvataggio del batch (append, non sovrascrive) ---
    df_quote_batch = pd.DataFrame(quote_batch)
    header_quote = not os.path.exists(PATH_QUOTE_OUTPUT)
    df_quote_batch.to_csv(PATH_QUOTE_OUTPUT, mode='a', header=header_quote, index=False)

    if campioni_batch:
        df_campione_batch = pd.concat(campioni_batch, ignore_index=True)
        header_campione = not os.path.exists(PATH_CAMPIONE_OUTPUT)
        df_campione_batch.to_csv(PATH_CAMPIONE_OUTPUT, mode='a', header=header_campione, index=False)

    # --- Aggiornamento checkpoint ---
    comuni_processati_ora = comuni_gia_fatti + min((batch_idx + 1) * BATCH_SIZE, len(df_comuni_da_fare))
    with open(PATH_CHECKPOINT, 'w') as f:
        f.write(str(comuni_processati_ora))

    print(f"Batch {batch_idx + 1}/{n_batch_totali} — "
          f"comuni totali processati: {comuni_processati_ora}/{len(df_comuni)}")

print(f"\n✅ COMPLETATO — output finali:")
print(f"   Proporzioni per comune: {PATH_QUOTE_OUTPUT}")
print(f"   Campione individuale per R_i: {PATH_CAMPIONE_OUTPUT}")

# Pulizia checkpoint solo a lavoro concluso (così una disconnessione a metà non
# fa perdere il progresso: il checkpoint resta finché non arrivi in fondo)
if comuni_processati_ora >= len(df_comuni):
    os.remove(PATH_CHECKPOINT)
    print("   Checkpoint rimosso: lavoro concluso con successo.")

Comuni caricati: 7904
✅ Tutti i 7904 comuni hanno una corrispondenza in POSAS (età singola).
✅ Tutti i 7904 comuni hanno una regione armonizzata correttamente.

Distribuzione comuni per regione armonizzata:
Region_Harmonized
Lombardia                1506
Piemonte                 1181
Veneto                    563
Campania                  550
Calabria                  404
Sicilia                   391
Lazio                     378
Sardegna                  377
Emilia-Romagna            330
Abruzzo                   305
Trentino-Alto Adige       282
Toscana                   273
Puglia                    257
Liguria                   234
Marche                    225
Friuli-Venezia Giulia     215
Molise                    136
Basilicata                131
Umbria                     92
Valle d’Aosta              74
Name: count, dtype: int64

Scarto medio |somma frequenze reddito - Numero contribuenti|: 130.3
(uno scarto positivo è atteso per le celle oscurate MEF sotto soglia privacy)

✅

In [ ]:
df_quote_check = pd.read_csv('/content/drive/MyDrive/Paper ASMBI/Dataset/quota_istruzione_comuni.csv')
print(f"Righe in quota_istruzione_comuni.csv: {len(df_quote_check)}")
print(f"Comuni ancora mancanti rispetto a 7.904: {7904 - len(df_quote_check)}")

df_campione_check = pd.read_csv('/content/drive/MyDrive/Paper ASMBI/Dataset/campione_individuale_r_i.csv')
print(f"Righe nel campione individuale: {len(df_campione_check):,}")

Righe in quota_istruzione_comuni.csv: 7904
Comuni ancora mancanti rispetto a 7.904: 0
Righe nel campione individuale: 1,174,592
